# ⚙️ ReAct 에이전트 루프 밑바닥부터 구현하기

In [1]:
import re

# 가상의 도구(Tool)들을 파이썬 함수로 정의합니다.
def calculate(expression: str) -> str:
    """문자열 형태의 수식을 받아 파이썬 eval()로 계산합니다."""
    try:
        # 안전장치 없이 실습용으로 eval 사용
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

def get_word_length(word: str) -> str:
    """단어의 글자 수를 반환합니다."""
    return str(len(word))

# 도구들을 딕셔너리로 묶어 엔진이 참조할 수 있게 합니다.
tools = {
    "Calculate": calculate,
    "WordLength": get_word_length
}

## 1. 에이전트 루프 파서(Parser)
모델이 뱉어낸 텍스트에서 `Action: 함수이름`과 `Action Input: 인자`를 정규표현식으로 추출하는 파서입니다. LangChain 내부에서도 동일한 파싱 로직을 씁니다.

In [2]:
def parse_action(text: str):
    # 정규표현식으로 Action과 Action Input을 찾음
    action_pattern = r"Action: (.*?)\n"
    input_pattern = r"Action Input: (.*?)$" # 끝까지 매칭 (간소화)
    
    action_match = re.search(action_pattern, text)
    input_match = re.search(input_pattern, text)
    
    if action_match and input_match:
        return action_match.group(1).strip(), input_match.group(1).strip()
    return None, None

## 2. 수동(Manual) ReAct 루프 시뮬레이션
본래 LLM이 이 텍스트를 뱉어내야 하지만, 알고리즘 이해를 위해 '모델이 이 텍스트를 출력했다고 가정'하고 루프를 돌려봅니다.

In [3]:
print("👨‍💻 [사용자 질문]: 'antigravity'라는 단어의 글자 수에 5를 곱하면 얼마인가요?\n")

# Step 1: 모델의 첫 번째 응답 흉내내기
llm_response_1 = """Thought: 사용자가 특정 단어의 글자 수를 구하고 5를 곱하길 원한다. 먼저 글자 수를 알아내야겠다.
Action: WordLength
Action Input: antigravity"""

print("🤖 [LLM 출력 1]\n" + llm_response_1 + "\n")

# 엔진이 텍스트를 가로채어 파싱 후 함수 실행
action_name, action_input = parse_action(llm_response_1)
if action_name in tools:
    print(f"⚙️ [에이전트 엔진]: 함수 '{action_name}'을(를) '{action_input}' 인자로 실행합니다.")
    observation_1 = tools[action_name](action_input)
    print(f"👀 [관찰 결과]: {observation_1}\n")
    
# Step 2: 관찰 결과를 받은 모델의 두 번째 응답 흉내내기
llm_response_2 = f"""Observation: {observation_1}
Thought: 글자 수는 {observation_1}이구나. 이제 여기에 5를 곱해야겠다.
Action: Calculate
Action Input: {observation_1} * 5"""

print("🤖 [LLM 출력 2]\n" + llm_response_2 + "\n")

# 두 번째 행동 실행
action_name, action_input = parse_action(llm_response_2)
if action_name in tools:
    print(f"⚙️ [에이전트 엔진]: 함수 '{action_name}'을(를) '{action_input}' 인자로 실행합니다.")
    observation_2 = tools[action_name](action_input)
    print(f"👀 [관찰 결과]: {observation_2}\n")

# Step 3: 최종 정답 도출
llm_response_3 = f"""Observation: {observation_2}
Thought: 계산이 끝났다. 사용자에게 정답을 알려줄 수 있다.
Final Answer: 'antigravity'의 글자 수는 11글자이며, 여기에 5를 곱하면 55가 됩니다."""

print("🤖 [LLM 최종 출력]\n" + llm_response_3)

print("\n💡 결론: LLM은 한 번에 정답을 찍는 것이 아니라, 텍스트 파싱과 파이썬 함수의 연결 고리를 통해 '생각➔행동➔관찰'의 무한 While 루프를 돌며 자율적으로 미션을 완수합니다!")

👨‍💻 [사용자 질문]: 'antigravity'라는 단어의 글자 수에 5를 곱하면 얼마인가요?

🤖 [LLM 출력 1]
Thought: 사용자가 특정 단어의 글자 수를 구하고 5를 곱하길 원한다. 먼저 글자 수를 알아내야겠다.
Action: WordLength
Action Input: antigravity

⚙️ [에이전트 엔진]: 함수 'WordLength'을(를) 'antigravity' 인자로 실행합니다.
👀 [관찰 결과]: 11

🤖 [LLM 출력 2]
Observation: 11
Thought: 글자 수는 11이구나. 이제 여기에 5를 곱해야겠다.
Action: Calculate
Action Input: 11 * 5

⚙️ [에이전트 엔진]: 함수 'Calculate'을(를) '11 * 5' 인자로 실행합니다.
👀 [관찰 결과]: 55

🤖 [LLM 최종 출력]
Observation: 55
Thought: 계산이 끝났다. 사용자에게 정답을 알려줄 수 있다.
Final Answer: 'antigravity'의 글자 수는 11글자이며, 여기에 5를 곱하면 55가 됩니다.

💡 결론: LLM은 한 번에 정답을 찍는 것이 아니라, 텍스트 파싱과 파이썬 함수의 연결 고리를 통해 '생각➔행동➔관찰'의 무한 While 루프를 돌며 자율적으로 미션을 완수합니다!
